# SkinAI — DS_unified 통합 모델 학습

**목적**: DS14(3종) + DS15(8종) AI Hub 합성 데이터와 3개 외부 임상 데이터셋을 혼합 학습해  
실제 이미지 도메인 갭을 최소화한 11종 통합 분류 모델 생성

| 데이터 | 이미지 수 | 클래스 | idx | 용도 |
|--------|----------|--------|-----|------|
| AI Hub DS14 합성 (3종) | 4,320 train / 480 test / 600 val | 건선·아토피피부염·여드름 | 0~2 | 학습 |
| AI Hub DS15 합성 (8종) | 5,760 train / 640 test / 800 val | ISIC 매핑 8종 | 3~10 | 학습 |
| DermNet NZ | 3,373 train / 1,096 test | 건선·아토피·여드름 | 0~2 | 학습 + 홀드아웃 |
| ISIC 2019 | 11,569 train / 2,042 val | ISIC 매핑 8종 | 3~10 | 학습 + 홀드아웃 |
| HAM10000 | 5,363 train / 947 val | ISIC 매핑 7종 (SCC 제외) | 3~10 | 학습 + 홀드아웃 |

**unified 데이터셋 3-way split** (DS14+DS15 통합 기준):
- `train.csv`: 10,080장 (DS14 4,320 + DS15 5,760) — 학습 전용
- `val.csv`: 1,400장 (DS14 600 + DS15 800) — 에폭 선택용 (best epoch 결정)
- `test.csv`: 1,120장 (train 10% 층화 분리) — 최종 성능 측정용 (학습 중 미사용)

> ⚠️ 편평세포암(SCC, idx=8)은 ISIC 2019 94장만 존재 — AI Hub 합성 중심으로 학습됩니다.

In [34]:
# GPU / RAM 확인
!nvidia-smi
from psutil import virtual_memory
ram_gb = virtual_memory().total / 1e9
print(f"시스템 RAM: {ram_gb:.1f} GB")

Sat May  9 12:16:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   46C    P8             17W /   72W |       3MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [35]:
# ── 셀 1: 환경 감지 ────────────────────────────────────────────
import os
from pathlib import Path

try:
    import google.colab
    IS_COLAB = True
    COLAB_ROOT = "/content/colab_skin_ai"
    PROJECT_ROOT = COLAB_ROOT
except ImportError:
    IS_COLAB = False
    PROJECT_ROOT = str(Path.cwd())

print(f"환경        : {'Google Colab' if IS_COLAB else '로컬'}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")

환경        : Google Colab
PROJECT_ROOT: /content/colab_skin_ai


In [36]:
# ── 셀 2: Drive 마운트 (Colab 전용) ────────────────────────────
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = "/content/drive/MyDrive/skin_ai"
else:
    print("로컬 환경 — Drive 마운트 건너뜀")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [37]:
# ── 셀 3: 소스코드 clone / pull (Colab 전용) ────────────────────
if IS_COLAB:
    from dotenv import load_dotenv

    _env_path = f"{DRIVE_ROOT}/.env"
    if Path(_env_path).exists():
        load_dotenv(_env_path)
    else:
        print(f"경고: {_env_path} 없음 — GITHUB_TOKEN 없이 시도합니다")

    _token = os.getenv("GITHUB_TOKEN", "")
    _repo_url = (
        f"https://{_token}@github.com/kyoe-23/skin_ai.git"
        if _token else
        "https://github.com/kyoe-23/skin_ai.git"
    )

    if not Path(COLAB_ROOT).exists():
        !git clone {_repo_url} {COLAB_ROOT}
    else:
        # pull 전 결과 파일(untracked) 제거 — git에 커밋된 파일과 충돌 방지
        !git -C {COLAB_ROOT} clean -fd ai/results/
        !git -C {COLAB_ROOT} pull
else:
    print("로컬 환경 — 클론 건너뜀")

Already up to date.


In [42]:
# ── 셀 4: 프로젝트 루트 이동 + 데이터 심링크 설정 ────────────────
os.chdir(PROJECT_ROOT)
print(f"현재 디렉토리: {os.getcwd()}")

if IS_COLAB:
    os.makedirs("data/processed", exist_ok=True)

    def _symlink(src: str, dst: str, label: str):
        src_path = Path(src)
        dst_path = Path(dst)
        if src_path.exists():
            if dst_path.is_symlink():
                dst_path.unlink()
            if not dst_path.exists():
                os.symlink(src_path, dst_path)
            print(f"  ✅ {dst} 심링크 완료")
        else:
            print(f"  ❌ {label} 없음 — {src}")

    # AI Hub 원본 ZIP (AihubFacialDataset 사용)
    _symlink(f"{DRIVE_ROOT}/data/dataset_14", "data/dataset_14", "dataset_14")
    _symlink(f"{DRIVE_ROOT}/data/dataset_15", "data/dataset_15", "dataset_15")

    # 통합 전처리 CSV (DS14 3종 + DS15 8종, idx 0~10)
    _symlink(f"{DRIVE_ROOT}/data/processed/unified",  "data/processed/unified",  "unified CSV")

    # 외부 임상 전처리 CSV (zip_path 형식 — ZIP 방식)
    _symlink(f"{DRIVE_ROOT}/data/processed/dermnet",  "data/processed/dermnet",  "DermNet CSV")
    _symlink(f"{DRIVE_ROOT}/data/processed/isic2019", "data/processed/isic2019", "ISIC CSV")
    _symlink(f"{DRIVE_ROOT}/data/processed/ham10000", "data/processed/ham10000", "HAM10000 CSV")

    # 외부 임상 ZIP 파일 (이미지 폴더 대신 ZIP 심링크 — Drive I/O 최적화)
    _symlink(f"{DRIVE_ROOT}/data/dermnet.zip",  "data/dermnet.zip",  "DermNet ZIP")
    _symlink(f"{DRIVE_ROOT}/data/isic2019.zip", "data/isic2019.zip", "ISIC 2019 ZIP")
    _symlink(f"{DRIVE_ROOT}/data/ham10000.zip", "data/ham10000.zip", "HAM10000 ZIP")

else:
    for d in [
        "data/dataset_14", "data/dataset_15",
        "data/processed/unified",
        "data/processed/dermnet", "data/processed/isic2019", "data/processed/ham10000",
        "data/dermnet.zip", "data/isic2019.zip", "data/ham10000.zip",
    ]:
        status = "✅" if Path(d).exists() else "❌"
        print(f"{status} {d}")

!ls data/

현재 디렉토리: /content/colab_skin_ai
  ✅ data/dataset_14 심링크 완료
  ✅ data/dataset_15 심링크 완료
  ✅ data/processed/unified 심링크 완료
  ✅ data/processed/dermnet 심링크 완료
  ✅ data/processed/isic2019 심링크 완료
  ✅ data/processed/ham10000 심링크 완료
  ✅ data/dermnet.zip 심링크 완료
  ✅ data/isic2019.zip 심링크 완료
  ✅ data/ham10000.zip 심링크 완료
dataset_14  dataset_15	dermnet.zip  ham10000.zip  isic2019.zip  processed


In [43]:
# ── 셀 5: 패키지 설치 ───────────────────────────────────────────
!pip install -q \
    torch torchvision \
    pandas pillow tqdm \
    matplotlib python-dotenv scikit-learn

In [44]:
# ── 셀 5-B: Drive에서 체크포인트 복원 (재평가 전용) ────────────
# 학습 없이 재평가만 할 때 — Drive에 저장된 best.pth/training_log.json/eval_*/ 를
# Colab 작업 디렉토리로 복사. 학습부터 새로 시작하면 이 셀은 건너뛰어도 됨.
if IS_COLAB:
    import shutil
    from pathlib import Path

    CKPT_SRC = f"{DRIVE_ROOT}/ai/results/DS_unified"
    CKPT_DST = f"{COLAB_ROOT}/ai/results/DS_unified"

    if Path(CKPT_SRC).exists():
        shutil.copytree(CKPT_SRC, CKPT_DST, dirs_exist_ok=True)
        pth = Path(f"{CKPT_DST}/checkpoint/best.pth")
        print(f"{'✅' if pth.exists() else '❌'} best.pth 복원: {pth}")
    else:
        print(f"❌ Drive에 체크포인트 없음: {CKPT_SRC}")
else:
    print("로컬 환경 — 복원 불필요 (이미 ai/results/DS_unified/ 에 존재)")

✅ best.pth 복원: /content/colab_skin_ai/ai/results/DS_unified/checkpoint/best.pth


In [45]:
# ── 셀 6: 학습 전 체크리스트 ────────────────────────────────────
import json
import torch
import pandas as pd
from pathlib import Path

checks = []
warnings = []

# 1. GPU
gpu_ok = torch.cuda.is_available()
checks.append(("GPU 사용 가능", gpu_ok))
if gpu_ok:
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  GPU: {name} ({vram:.0f} GB)")

# 2. AI Hub 원본 ZIP
for tag, path in [("DS14 원본", "data/dataset_14"), ("DS15 원본", "data/dataset_15")]:
    ok = Path(path).exists()
    checks.append((f"{tag} ({path})", ok))

# 3. 통합 전처리 CSV + metadata
unified_dir = Path("data/processed/unified")
unified_train = unified_dir / "train.csv"
unified_ok = unified_train.exists()
checks.append(("통합 unified CSV (data/processed/unified)", unified_ok))

unified_meta = unified_dir / "metadata.json"
meta_classes = []
if unified_meta.exists():
    with open(unified_meta, encoding="utf-8") as f:
        meta = json.load(f)
    meta_num = meta.get("num_classes", 0)
    meta_classes = sorted(meta.get("classes", {}), key=lambda k: meta["classes"][k])
    checks.append((f"unified metadata.json (num_classes={meta_num})", meta_num == 11))
else:
    checks.append(("unified metadata.json", False))

if unified_ok:
    df = pd.read_csv(unified_train)
    print(f"  unified train {len(df):,}건 | 클래스: {df['class_name'].nunique()}종")

# 4. 외부 임상 CSV
for tag, path in [
    ("DermNet CSV",  "data/processed/dermnet/train.csv"),
    ("ISIC 2019 CSV", "data/processed/isic2019/train.csv"),
    ("HAM10000 CSV",  "data/processed/ham10000/train.csv"),
]:
    ok = Path(path).exists()
    checks.append((tag, ok))
    if ok:
        n = len(pd.read_csv(path))
        print(f"  {tag}: {n:,}건")

# 5. 외부 임상 ZIP (이미지 폴더 대신 ZIP 사용 — Drive I/O 최적화)
for tag, path in [
    ("DermNet ZIP",  "data/dermnet.zip"),
    ("ISIC 2019 ZIP", "data/isic2019.zip"),
    ("HAM10000 ZIP",  "data/ham10000.zip"),
]:
    ok = Path(path).exists()
    checks.append((tag, ok))

# 학습 범위 안내
warnings.append(f"  ℹ️  DS_unified 11종: {', '.join(meta_classes) if meta_classes else '(metadata.json 부재)'}")
warnings.append("  ⚠️  편평세포암(SCC) HAM10000 미포함 — ISIC 94장으로만 실사 학습")
warnings.append("  ⚠️  DS14 6종 중 주사·지루피부염·정상 3종은 외부 임상 데이터 미확보로 unified에서 제외")
warnings.append("  ⚠️  DS15 15종 중 보웬병·비립종·사마귀·표피낭종·피지샘증식증·화농 육아종·흑색점 7종도 동일 사유로 제외")

print("\n" + "=" * 60)
print("학습 전 체크리스트")
print("=" * 60)
all_ok = True
for name, ok in checks:
    status = "✅" if ok else "❌"
    print(f"{status} {name}")
    if not ok:
        all_ok = False

print("\n주의사항:")
for w in warnings:
    print(w)

print("\n" + ("🟢 모든 필수 조건 충족" if all_ok else "🔴 위 항목을 먼저 해결하세요"))

  GPU: NVIDIA L4 (24 GB)
  unified train 10,080건 | 클래스: 11종
  DermNet CSV: 3,373건
  ISIC 2019 CSV: 11,569건
  HAM10000 CSV: 5,363건

학습 전 체크리스트
✅ GPU 사용 가능
✅ DS14 원본 (data/dataset_14)
✅ DS15 원본 (data/dataset_15)
✅ 통합 unified CSV (data/processed/unified)
✅ unified metadata.json (num_classes=11)
✅ DermNet CSV
✅ ISIC 2019 CSV
✅ HAM10000 CSV
✅ DermNet ZIP
✅ ISIC 2019 ZIP
✅ HAM10000 ZIP

주의사항:
  ℹ️  DS_unified 11종: 건선, 아토피피부염, 여드름, 광선각화증, 기저세포암, 멜라닌세포모반, 악성흑색종, 지루각화증, 편평세포암, 피부섬유종, 혈관종
  ⚠️  편평세포암(SCC) HAM10000 미포함 — ISIC 94장으로만 실사 학습
  ⚠️  DS14 6종 중 주사·지루피부염·정상 3종은 외부 임상 데이터 미확보로 unified에서 제외
  ⚠️  DS15 15종 중 보웬병·비립종·사마귀·표피낭종·피지샘증식증·화농 육아종·흑색점 7종도 동일 사유로 제외

🟢 모든 필수 조건 충족


In [17]:
# ── 셀 7: DS_unified 통합 학습 실행 ─────────────────────────────
# AI Hub 합성 11,200장 + DermNet 3,373장 + ISIC 11,569장 + HAM10000 5,363장 혼합
# WeightedRandomSampler: AI Hub weight=1.0, 외부 3종 weight=1.5
# 체크포인트 저장: ai/results/DS_unified/checkpoint/
CKPT_DIR = "ai/results/DS_unified/checkpoint"
RESUME_CKPT = f"{CKPT_DIR}/best.pth"

from pathlib import Path
_resume_flag = f"--resume {RESUME_CKPT}" if Path(RESUME_CKPT).exists() else ""
print(f"Resume: {_resume_flag or '없음 (처음부터)'}")

!EXTRA_DATA_DIR="data/processed/dermnet data/processed/isic2019 data/processed/ham10000" \
 EXTERNAL_WEIGHT=1.5 \
 EXPERIMENT_NAME=ds_unified \
 CHECKPOINT_DIR={CKPT_DIR} \
 python -m ai.training.classifier.train \
     --backbone densenet121 \
     --data_dir data/processed/unified \
     --num_classes 11 \
     --num_epochs 100 \
     --batch_size 64 \
     --root_dir {PROJECT_ROOT} \
     {_resume_flag}

Resume: 없음 (처음부터)
INFO [INFO] CUDA 사용
피부질환 분류 모델 학습
  backbone    : densenet121
  device      : cuda
  epochs      : 100
  batch       : 64
  lr          : 0.0005
  warmup      : 3 epochs
  scheduler   : CosineAnnealingLR (T_max=97)
  num_classes : 11
  data_dir    : data/processed/unified
INFO Dataset 로드: data/processed/unified/train.csv (10080건, direction=None)
INFO Dataset 로드: data/processed/unified/val.csv (1400건, direction=None)
INFO ExternalFacialDataset 로드(ZIP): data/processed/dermnet/train.csv (3373건)
INFO ExternalFacialDataset 로드(ZIP): data/processed/dermnet/val.csv (596건)
INFO ExternalFacialDataset 로드(ZIP): data/processed/isic2019/train.csv (11569건)
INFO ExternalFacialDataset 로드(ZIP): data/processed/isic2019/val.csv (2042건)
INFO ExternalFacialDataset 로드(ZIP): data/processed/ham10000/train.csv (5363건)
INFO ExternalFacialDataset 로드(ZIP): data/processed/ham10000/val.csv (947건)

  Train: 30385건 (AI Hub 10080 + 외부 20305)
  Val  : 4985건
INFO   Model     : DenseNet
INFO   Total     

In [46]:
# ── 셀 8: 평가 1 — unified val (합성 성능 유지 확인) ─────────────
# DS14 3종 + DS15 8종 통합 val (1,400장) — 합성 val 성능 유지 확인
print("=" * 60)
print("평가 1: unified val (합성 기준선 확인)")
print("=" * 60)
!python -m ai.testing.evaluate \
    --checkpoint ai/results/DS_unified/checkpoint/best.pth \
    --data_dir data/processed/unified \
    --split val \
    --output_dir ai/results/DS_unified/eval_unified_val \
    --root_dir {PROJECT_ROOT}

평가 1: unified val (합성 기준선 확인)
INFO [INFO] CUDA 사용
INFO Dataset 로드: data/processed/unified/val.csv (1400건, direction=None)
평가 데이터: val.csv (1400건)

 SkinAI 분류 모델 평가 결과
 모델: densenet121
------------------------------------------------------------
 Top-1 Accuracy : 96.21%  가이드라인 목표(80%): 달성
 Top-3 Accuracy : 99.93%
 Macro F1-Score : 0.9741
 Macro AUC      : 0.9981
------------------------------------------------------------
 클래스              Prec   Recall       F1      AUC
------------------------------------------------------------
 건선             0.8947   0.9350   0.9144   0.9948
 아토피피부염         0.9146   0.9100   0.9123   0.9952
 여드름            0.9479   0.9100   0.9286   0.9908
 광선각화증          1.0000   0.9900   0.9950   1.0000
 기저세포암          0.9901   1.0000   0.9950   0.9998
 멜라닌세포모반        0.9900   0.9900   0.9900   1.0000
 악성흑색종          0.9899   0.9800   0.9849   0.9985
 지루각화증          1.0000   1.0000   1.0000   1.0000
 편평세포암          0.9901   1.0000   0.9950   1.0000
 피부섬유종        

In [47]:
# ── 셀 8-B: 평가 1-B — unified test (학습 중 미노출 홀드아웃) ─────────
# DS14 3종 + DS15 8종 train의 10%를 층화 분리한 holdout test (1,120장)
# 학습 내내 한 번도 쓰지 않은 데이터 → 진짜 합성 도메인 성능 수치
print("=" * 60)
print("평가 1-B: unified test 홀드아웃 (합성 도메인 진짜 성능)")
print("train 10% 층화 분리 — 1,120장, 11종 클래스별 80~160장")
print("=" * 60)
!python -m ai.testing.evaluate \
    --checkpoint ai/results/DS_unified/checkpoint/best.pth \
    --data_dir data/processed/unified \
    --split test \
    --output_dir ai/results/DS_unified/eval_unified_test \
    --root_dir {PROJECT_ROOT}

평가 1-B: unified test 홀드아웃 (합성 도메인 진짜 성능)
train 10% 층화 분리 — 1,120장, 11종 클래스별 80~160장
INFO [INFO] CUDA 사용
INFO Dataset 로드: data/processed/unified/test.csv (1120건, direction=None)
평가 데이터: test.csv (1120건)

 SkinAI 분류 모델 평가 결과
 모델: densenet121
------------------------------------------------------------
 Top-1 Accuracy : 97.59%  가이드라인 목표(80%): 달성
 Top-3 Accuracy : 100.00%
 Macro F1-Score : 0.9841
 Macro AUC      : 0.9994
------------------------------------------------------------
 클래스              Prec   Recall       F1      AUC
------------------------------------------------------------
 건선             0.9669   0.9125   0.9389   0.9968
 아토피피부염         0.9554   0.9375   0.9464   0.9967
 여드름            0.9186   0.9875   0.9518   0.9993
 광선각화증          1.0000   1.0000   1.0000   1.0000
 기저세포암          0.9877   1.0000   0.9938   1.0000
 멜라닌세포모반        1.0000   1.0000   1.0000   1.0000
 악성흑색종          1.0000   1.0000   1.0000   1.0000
 지루각화증          1.0000   1.0000   1.0000   1.0000
 편평세포암 

In [48]:
# ── 셀 9: 평가 2 — DermNet test (건선·아토피·여드름 실사 검증) ──────
# DermNet test.csv 1,096장 — 홀드아웃 세트
# 참고: DS14_mixed DermNet test Top-1 82.30% (baseline 35.77%)
print("=" * 60)
print("평가 2: DermNet test 홀드아웃 (건선·아토피·여드름 도메인 갭)")
print("DS14_mixed 기준 → 82.30%")
print("=" * 60)
!python -m ai.testing.evaluate \
    --checkpoint ai/results/DS_unified/checkpoint/best.pth \
    --data_dir data/processed/dermnet \
    --split test \
    --output_dir ai/results/DS_unified/eval_dermnet_test

평가 2: DermNet test 홀드아웃 (건선·아토피·여드름 도메인 갭)
DS14_mixed 기준 → 82.30%
INFO [INFO] CUDA 사용
INFO Dataset 로드: data/processed/dermnet/test.csv (1096건, direction=None)
평가 데이터: test.csv (1096건)

 SkinAI 분류 모델 평가 결과
 모델: densenet121
------------------------------------------------------------
 Top-1 Accuracy : 82.48%  가이드라인 목표(80%): 달성
 Top-3 Accuracy : 99.54%
 Macro F1-Score : 0.2258
 Macro AUC      : 0.0000
------------------------------------------------------------
 클래스              Prec   Recall       F1      AUC
------------------------------------------------------------
 건선             0.7645   0.7472   0.7557   0.9123
 아토피피부염         0.8254   0.7986   0.8118   0.9215
 여드름            0.8862   0.9487   0.9164   0.9804
 광선각화증          0.0000   0.0000   0.0000   0.0000
 기저세포암          0.0000   0.0000   0.0000   0.0000
 멜라닌세포모반        0.0000   0.0000   0.0000   0.0000
 악성흑색종          0.0000   0.0000   0.0000   0.0000
 지루각화증          0.0000   0.0000   0.0000   0.0000
 편평세포암          0.0000   0

In [49]:
# ── 셀 10: 평가 3 — ISIC 2019 val (악성 클래스 도메인 갭) ───────────
# ISIC 2019 val.csv 2,042장 — 홀드아웃 세트
# 참고: DS15_mixed ISIC val Top-1 77.72% (baseline 26.40%)
print("=" * 60)
print("평가 3: ISIC 2019 val 홀드아웃 (악성 8종 도메인 갭)")
print("DS15_mixed 기준 → Top-1 77.72% | Weighted F1 0.7764")
print("=" * 60)
!python -m ai.testing.evaluate \
    --checkpoint ai/results/DS_unified/checkpoint/best.pth \
    --data_dir data/processed/isic2019 \
    --split val \
    --output_dir ai/results/DS_unified/eval_isic_val

평가 3: ISIC 2019 val 홀드아웃 (악성 8종 도메인 갭)
DS15_mixed 기준 → Top-1 77.72% | Weighted F1 0.7764
INFO [INFO] CUDA 사용
INFO Dataset 로드: data/processed/isic2019/val.csv (2042건, direction=None)
평가 데이터: val.csv (2042건)

 SkinAI 분류 모델 평가 결과
 모델: densenet121
------------------------------------------------------------
 Top-1 Accuracy : 80.07%  가이드라인 목표(80%): 달성
 Top-3 Accuracy : 96.33%
 Macro F1-Score : 0.5602
 Macro AUC      : 0.0000
------------------------------------------------------------
 클래스              Prec   Recall       F1      AUC
------------------------------------------------------------
 건선             0.0000   0.0000   0.0000   0.0000
 아토피피부염         0.0000   0.0000   0.0000   0.0000
 여드름            0.0000   0.0000   0.0000   0.0000
 광선각화증          0.5389   0.6923   0.6061   0.9594
 기저세포암          0.8562   0.8867   0.8712   0.9804
 멜라닌세포모반        0.8201   0.8511   0.8353   0.9652
 악성흑색종          0.7967   0.7578   0.7768   0.9412
 지루각화증          0.8273   0.8147   0.8210   0.9547
 편평세

In [50]:
# ── 셀 11: 평가 4 — HAM10000 val (소수 클래스 보강 효과 확인) ────────
# HAM10000 val.csv 947장 — 혼합 후 소수 클래스(피부섬유종·혈관종·광선각화증) 개선 확인
print("=" * 60)
print("평가 4: HAM10000 val (소수 클래스 보강 효과)")
print("주목: 피부섬유종(ISIC 36장+115장), 혈관종(38+142), 광선각화증(130+327)")
print("=" * 60)
!python -m ai.testing.evaluate \
    --checkpoint ai/results/DS_unified/checkpoint/best.pth \
    --data_dir data/processed/ham10000 \
    --split val \
    --output_dir ai/results/DS_unified/eval_ham10000_val

평가 4: HAM10000 val (소수 클래스 보강 효과)
주목: 피부섬유종(ISIC 36장+115장), 혈관종(38+142), 광선각화증(130+327)
INFO [INFO] CUDA 사용
INFO Dataset 로드: data/processed/ham10000/val.csv (947건, direction=None)
평가 데이터: val.csv (947건)

 SkinAI 분류 모델 평가 결과
 모델: densenet121
------------------------------------------------------------
 Top-1 Accuracy : 88.07%  가이드라인 목표(80%): 달성
 Top-3 Accuracy : 99.26%
 Macro F1-Score : 0.5634
 Macro AUC      : 0.0000
------------------------------------------------------------
 클래스              Prec   Recall       F1      AUC
------------------------------------------------------------
 건선             0.0000   0.0000   0.0000   0.0000
 아토피피부염         0.0000   0.0000   0.0000   0.0000
 여드름            0.0000   0.0000   0.0000   0.0000
 광선각화증          0.8286   0.5918   0.6905   0.9728
 기저세포암          0.9189   0.8831   0.9007   0.9938
 멜라닌세포모반        0.9302   0.9178   0.9239   0.9774
 악성흑색종          0.8408   0.7904   0.8148   0.9699
 지루각화증          0.8652   0.9333   0.8980   0.9927
 편평세포암 

In [51]:
# ── 셀 12: 체크포인트 Drive 저장 (Colab 전용) ────────────────────
# 런타임 종료 전 반드시 실행 — 저장하지 않으면 학습 결과 소실
if IS_COLAB:
    import shutil
    from pathlib import Path

    CKPT_SRC = f"{COLAB_ROOT}/ai/results/DS_unified"
    CKPT_DST = f"{DRIVE_ROOT}/ai/results/DS_unified"

    if not Path(CKPT_SRC).exists():
        print(f"❌ 체크포인트 없음: {CKPT_SRC}")
    else:
        shutil.copytree(CKPT_SRC, CKPT_DST, dirs_exist_ok=True)
        print(f"✅ Drive 저장 완료: {CKPT_DST}")
else:
    print("로컬 환경 — 체크포인트 이미 ai/results/DS_unified/ 에 저장됨")

✅ Drive 저장 완료: /content/drive/MyDrive/skin_ai/ai/results/DS_unified
